# TP 0 — Understanding Regularization (Ridge, Lasso, Elastic Net)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/racousin/L2Math/blob/main/session2/tp_0_regularization.ipynb)

## Objectives
1. Understand why Ordinary Least Squares (OLS) can overfit when there are many features
2. Learn how Ridge regression (L2 penalty) shrinks coefficients toward zero
3. Learn how Lasso regression (L1 penalty) performs feature selection by zeroing out coefficients
4. Visualize the geometric intuition behind L1 vs L2 regularization
5. Understand Elastic Net as a combination of Ridge and Lasso
6. Compare all three methods on the same dataset

## Setup

Run the cell below to install and import the required packages.

In [ ]:
!pip install git+https://github.com/racousin/L2Math.git

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

from l2math import generate_linear_data, plot_coefficient_path

np.random.seed(42)
print("Setup complete!")

---
# Part 1: The Problem — OLS Can Overfit

## Ordinary Least Squares (OLS)

OLS minimizes the mean squared error:

$$\mathcal{L}_{\text{OLS}} = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

When the number of features is large relative to the number of samples, or when features are correlated (multicollinearity), OLS tends to **overfit**: it fits the training data very well but generalizes poorly to new data. The learned coefficients can become very large in magnitude as the model exploits noise in the training data.

Let’s see this in action with a dataset that has 20 features but only 5 truly informative ones.

In [ ]:
# Generate base data with 5 informative features
X_base, y, true_weights, true_bias = generate_linear_data(
    n_samples=80, n_features=5, noise=1.0, weight_scale=3.0, seed=42
)

# Add 15 pure noise features (not related to y)
np.random.seed(123)
X_noise = np.random.randn(80, 15) * 2
X = np.hstack([X_base, X_noise])
feature_names = [f"x{i}" for i in range(20)]
print(f"\nFull dataset: X shape = {X.shape}, y shape = {y.shape}")
print(f"Features 0-4 are informative, features 5-19 are noise")

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y.ravel(), test_size=0.3, random_state=42
)

# Scale features (important for regularization)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Fit OLS (no regularization)
ols = LinearRegression()
ols.fit(X_train_scaled, y_train)

train_mse = mean_squared_error(y_train, ols.predict(X_train_scaled))
test_mse = mean_squared_error(y_test, ols.predict(X_test_scaled))

print(f"\nOLS Results:")
print(f"  Train MSE: {train_mse:.4f}")
print(f"  Test MSE:  {test_mse:.4f}")
print(f"  Gap:       {test_mse - train_mse:.4f}")
print(f"\nCoefficients (top 10 by magnitude):")
sorted_idx = np.argsort(np.abs(ols.coef_))[::-1]
for i in sorted_idx[:10]:
    tag = " (informative)" if i < 5 else " (noise)"
    print(f"  {feature_names[i]:>3}: {ols.coef_[i]:+.4f}{tag}")

**Observation:** OLS assigns large coefficients to noise features, leading to a significant gap between train and test MSE. The model is **overfitting** — it captures noise in the training data as if it were signal.

**Solution:** Add a **penalty term** to the loss function that discourages large coefficients. This is called **regularization**.

---
# Part 2: Ridge Regression (L2 Penalty)

Ridge regression adds an **L2 penalty** (sum of squared coefficients) to the loss:

$$\mathcal{L}_{\text{Ridge}} = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2 + \alpha \sum_{j=1}^{p} w_j^2$$

- $\alpha > 0$ controls the regularization strength
- **Small $\alpha$**: little regularization → close to OLS
- **Large $\alpha$**: strong regularization → coefficients shrink toward zero

**Key property:** Ridge shrinks all coefficients toward zero but **never sets them exactly to zero**. All features remain in the model.

In [ ]:
alphas = [0.01, 0.1, 1, 10, 100]

print(f"{'Alpha':>8} | {'Train MSE':>10} | {'Test MSE':>10} | {'Max |coef|':>12}")
print("-" * 50)

for alpha in alphas:
    ridge = Ridge(alpha=alpha)
    ridge.fit(X_train_scaled, y_train)
    tr_mse = mean_squared_error(y_train, ridge.predict(X_train_scaled))
    te_mse = mean_squared_error(y_test, ridge.predict(X_test_scaled))
    max_coef = np.max(np.abs(ridge.coef_))
    print(f"{alpha:>8.2f} | {tr_mse:>10.4f} | {te_mse:>10.4f} | {max_coef:>12.4f}")

In [ ]:
# Sweep 100 alpha values and collect coefficients
alphas_path = np.logspace(-2, 4, 100)
ridge_coefs = []

for alpha in alphas_path:
    ridge = Ridge(alpha=alpha)
    ridge.fit(X_train_scaled, y_train)
    ridge_coefs.append(ridge.coef_)

ridge_coefs = np.array(ridge_coefs)

plot_coefficient_path(alphas_path, ridge_coefs, title="Ridge — Coefficient Path")
plt.show()

**Observations:**
- As $\alpha$ increases, all coefficients **smoothly shrink toward zero**.
- No coefficient is ever exactly zero — Ridge does not perform feature selection.
- At small $\alpha$, the coefficients are close to the OLS solution (some may be large and noisy).
- At large $\alpha$, all coefficients are very small, and the model underfits.

---
# Part 3: Lasso Regression (L1 Penalty)

Lasso uses an **L1 penalty** (sum of absolute values) instead:

$$\mathcal{L}_{\text{Lasso}} = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2 + \alpha \sum_{j=1}^{p} |w_j|$$

**Key property:** Lasso can set coefficients **exactly to zero**, effectively performing **feature selection**. This happens because the L1 constraint region has sharp corners where the solution naturally lands.

## Geometric Intuition

In 2D, the constraint region for L2 is a **circle** ($w_1^2 + w_2^2 \leq t$) while for L1 it is a **diamond** ($|w_1| + |w_2| \leq t$). The optimal solution is where the elliptical MSE contours first touch the constraint region:
- The **circle** has no corners → contours typically touch on a smooth curve → both coefficients are non-zero
- The **diamond** has **corners on the axes** → contours are more likely to touch at a corner → one coefficient is exactly zero

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Elliptical contours centered at OLS solution (3, 0.5)
w1_ols, w2_ols = 3.0, 0.5
a, b = 1.0, 4.0  # control ellipse shape

w1 = np.linspace(-4, 5, 300)
w2 = np.linspace(-3, 3, 300)
W1, W2 = np.meshgrid(w1, w2)
Loss = a * (W1 - w1_ols)**2 + b * (W2 - w2_ols)**2

t = 1.5  # constraint size

# --- Left panel: L2 (circle) ---
theta = np.linspace(0, 2 * np.pi, 200)
circle_w1 = t * np.cos(theta)
circle_w2 = t * np.sin(theta)

ax = axes[0]
ax.contour(W1, W2, Loss, levels=np.linspace(1, 30, 12), cmap='RdYlBu_r', alpha=0.7)
ax.fill(circle_w1, circle_w2, alpha=0.2, color='blue')
ax.plot(circle_w1, circle_w2, 'b-', linewidth=2, label='L2 constraint region')
ax.plot(w1_ols, w2_ols, 'r*', markersize=15, label='OLS solution', zorder=5)
ax.axhline(y=0, color='k', linewidth=0.5, alpha=0.3)
ax.axvline(x=0, color='k', linewidth=0.5, alpha=0.3)
ax.set_xlabel('$w_1$', fontsize=12)
ax.set_ylabel('$w_2$', fontsize=12)
ax.set_title('Ridge (L2): Circle', fontsize=13)
ax.legend(fontsize=10, loc='lower left')
ax.set_aspect('equal')
ax.grid(True, alpha=0.2)

# --- Right panel: L1 (diamond) ---
diamond_w1 = [t, 0, -t, 0, t]
diamond_w2 = [0, t, 0, -t, 0]

ax = axes[1]
ax.contour(W1, W2, Loss, levels=np.linspace(1, 30, 12), cmap='RdYlBu_r', alpha=0.7)
ax.fill(diamond_w1, diamond_w2, alpha=0.2, color='orange')
ax.plot(diamond_w1, diamond_w2, color='orange', linewidth=2, label='L1 constraint region')
ax.plot(w1_ols, w2_ols, 'r*', markersize=15, label='OLS solution', zorder=5)
ax.axhline(y=0, color='k', linewidth=0.5, alpha=0.3)
ax.axvline(x=0, color='k', linewidth=0.5, alpha=0.3)
ax.set_xlabel('$w_1$', fontsize=12)
ax.set_ylabel('$w_2$', fontsize=12)
ax.set_title('Lasso (L1): Diamond — solution hits corner ($w_2=0$)', fontsize=13)
ax.legend(fontsize=10, loc='lower left')
ax.set_aspect('equal')
ax.grid(True, alpha=0.2)

plt.suptitle('Geometric Intuition: Why L1 Produces Sparse Solutions', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
alphas_lasso = [0.001, 0.01, 0.1, 0.5, 1.0]

print(f"{'Alpha':>8} | {'Train MSE':>10} | {'Test MSE':>10} | {'Non-zero coefs':>15}")
print("-" * 55)

for alpha in alphas_lasso:
    lasso = Lasso(alpha=alpha, max_iter=10000)
    lasso.fit(X_train_scaled, y_train)
    tr_mse = mean_squared_error(y_train, lasso.predict(X_train_scaled))
    te_mse = mean_squared_error(y_test, lasso.predict(X_test_scaled))
    n_nonzero = np.sum(lasso.coef_ != 0)
    print(f"{alpha:>8.3f} | {tr_mse:>10.4f} | {te_mse:>10.4f} | {n_nonzero:>15d} / 20")

In [ ]:
alphas_path_lasso = np.logspace(-3, 1, 100)
lasso_coefs = []

for alpha in alphas_path_lasso:
    lasso = Lasso(alpha=alpha, max_iter=10000)
    lasso.fit(X_train_scaled, y_train)
    lasso_coefs.append(lasso.coef_)

lasso_coefs = np.array(lasso_coefs)

plot_coefficient_path(alphas_path_lasso, lasso_coefs, title="Lasso — Coefficient Path")
plt.show()

**Observations:**
- As $\alpha$ increases, coefficients are driven to **exactly zero** one by one.
- At moderate $\alpha$, only the truly informative features (0–4) retain non-zero coefficients — Lasso performs **automatic feature selection**.
- At very large $\alpha$, all coefficients are zero and the model predicts the mean (underfitting).

---
# Part 4: Elastic Net (L1 + L2 Combined)

Elastic Net combines both penalties:

$$\mathcal{L}_{\text{ElasticNet}} = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2 + \alpha \left[ \rho \sum_{j=1}^{p} |w_j| + \frac{1-\rho}{2} \sum_{j=1}^{p} w_j^2 \right]$$

where:
- $\alpha$ controls the **overall** regularization strength
- $\rho$ (`l1_ratio` in sklearn) controls the **mix** between L1 and L2:
  - $\rho = 1$: pure Lasso
  - $\rho = 0$: pure Ridge
  - $0 < \rho < 1$: a blend of both

**When to use Elastic Net:**
- When you want feature selection (like Lasso) but also want to handle groups of correlated features (like Ridge)
- When Lasso selects only one feature from a correlated group, Elastic Net tends to select the whole group

In [ ]:
alpha_en = 0.1
l1_ratios = [0.1, 0.3, 0.5, 0.7, 0.9]

print(f"Elastic Net with alpha = {alpha_en}")
print(f"{'l1_ratio':>10} | {'Train MSE':>10} | {'Test MSE':>10} | {'Non-zero coefs':>15}")
print("-" * 55)

for l1r in l1_ratios:
    en = ElasticNet(alpha=alpha_en, l1_ratio=l1r, max_iter=10000)
    en.fit(X_train_scaled, y_train)
    tr_mse = mean_squared_error(y_train, en.predict(X_train_scaled))
    te_mse = mean_squared_error(y_test, en.predict(X_test_scaled))
    n_nonzero = np.sum(en.coef_ != 0)
    print(f"{l1r:>10.1f} | {tr_mse:>10.4f} | {te_mse:>10.4f} | {n_nonzero:>15d} / 20")

In [ ]:
alphas_path_en = np.logspace(-3, 1, 100)
en_coefs = []

for alpha in alphas_path_en:
    en = ElasticNet(alpha=alpha, l1_ratio=0.5, max_iter=10000)
    en.fit(X_train_scaled, y_train)
    en_coefs.append(en.coef_)

en_coefs = np.array(en_coefs)

plot_coefficient_path(alphas_path_en, en_coefs, title="Elastic Net (l1_ratio=0.5) — Coefficient Path")
plt.show()

**Observations:**
- Elastic Net is a **compromise** between Ridge and Lasso.
- Higher `l1_ratio` → more sparsity (closer to Lasso behavior).
- Lower `l1_ratio` → smoother shrinkage (closer to Ridge behavior).
- The coefficient path shows features being zeroed out, but less abruptly than pure Lasso.

---
# Part 5: Comparing All Three Methods

Let’s compare Ridge, Lasso, and Elastic Net side by side.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

alphas_comp = np.logspace(-2, 3, 100)

models = [
    ("Ridge (L2)", lambda a: Ridge(alpha=a)),
    ("Lasso (L1)", lambda a: Lasso(alpha=a, max_iter=10000)),
    ("Elastic Net (L1+L2)", lambda a: ElasticNet(alpha=a, l1_ratio=0.5, max_iter=10000)),
]

for ax, (title, Model) in zip(axes, models):
    coefs = []
    for alpha in alphas_comp:
        m = Model(alpha)
        m.fit(X_train_scaled, y_train)
        coefs.append(m.coef_)
    coefs = np.array(coefs)

    n_features = coefs.shape[1]
    colors = plt.cm.tab10(np.linspace(0, 1, min(n_features, 10)))
    for j in range(n_features):
        ax.plot(alphas_comp, coefs[:, j], linewidth=1.5,
                color=colors[j % len(colors)], alpha=0.7)

    ax.set_xscale("log")
    ax.set_xlabel(r"$\alpha$", fontsize=12)
    ax.set_ylabel("Coefficient value", fontsize=12)
    ax.set_title(title, fontsize=13)
    ax.axhline(y=0, color='k', linestyle='--', linewidth=0.5, alpha=0.5)
    ax.grid(True, alpha=0.3)

plt.suptitle("Coefficient Paths — Ridge vs Lasso vs Elastic Net", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
alphas_eval = np.logspace(-3, 3, 60)

mse_ridge = []
mse_lasso = []
mse_enet = []

for alpha in alphas_eval:
    ridge = Ridge(alpha=alpha)
    ridge.fit(X_train_scaled, y_train)
    mse_ridge.append(mean_squared_error(y_test, ridge.predict(X_test_scaled)))

    lasso = Lasso(alpha=alpha, max_iter=10000)
    lasso.fit(X_train_scaled, y_train)
    mse_lasso.append(mean_squared_error(y_test, lasso.predict(X_test_scaled)))

    en = ElasticNet(alpha=alpha, l1_ratio=0.5, max_iter=10000)
    en.fit(X_train_scaled, y_train)
    mse_enet.append(mean_squared_error(y_test, en.predict(X_test_scaled)))

plt.figure(figsize=(10, 5))
plt.plot(alphas_eval, mse_ridge, 'b-o', markersize=3, linewidth=2, label='Ridge')
plt.plot(alphas_eval, mse_lasso, 'r-s', markersize=3, linewidth=2, label='Lasso')
plt.plot(alphas_eval, mse_enet, 'g-^', markersize=3, linewidth=2, label='Elastic Net')
plt.xscale('log')
plt.xlabel(r'$\alpha$', fontsize=12)
plt.ylabel('Test MSE', fontsize=12)
plt.title('Test MSE vs Regularization Strength', fontsize=14)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Observations:**
- All three methods improve over OLS (at the right $\alpha$) by reducing overfitting.
- **Ridge** performs well with smooth coefficient shrinkage.
- **Lasso** achieves sparsity — useful when interpretability and feature selection matter.
- **Elastic Net** balances both behaviors.
- All methods have an optimal $\alpha$ — too small means overfitting, too large means underfitting.

In [ ]:
# Find best alpha for each method
results = {}

# Ridge
best_mse, best_alpha = min((m, a) for m, a in zip(mse_ridge, alphas_eval))
ridge_best = Ridge(alpha=best_alpha).fit(X_train_scaled, y_train)
results['Ridge'] = {
    'alpha': best_alpha,
    'test_mse': best_mse,
    'nonzero': np.sum(np.abs(ridge_best.coef_) > 1e-6)
}

# Lasso
best_mse, best_alpha = min((m, a) for m, a in zip(mse_lasso, alphas_eval))
lasso_best = Lasso(alpha=best_alpha, max_iter=10000).fit(X_train_scaled, y_train)
results['Lasso'] = {
    'alpha': best_alpha,
    'test_mse': best_mse,
    'nonzero': np.sum(np.abs(lasso_best.coef_) > 1e-6)
}

# Elastic Net
best_mse, best_alpha = min((m, a) for m, a in zip(mse_enet, alphas_eval))
en_best = ElasticNet(alpha=best_alpha, l1_ratio=0.5, max_iter=10000).fit(X_train_scaled, y_train)
results['Elastic Net'] = {
    'alpha': best_alpha,
    'test_mse': best_mse,
    'nonzero': np.sum(np.abs(en_best.coef_) > 1e-6)
}

# Print comparison table
print(f"{'Method':>12} | {'Best alpha':>12} | {'Test MSE':>10} | {'Non-zero coefs':>15}")
print("=" * 60)
for name, r in results.items():
    print(f"{name:>12} | {r['alpha']:>12.4f} | {r['test_mse']:>10.4f} | {r['nonzero']:>15d} / 20")
ols_mse = mean_squared_error(y_test, ols.predict(X_test_scaled))
print(f"{'OLS':>12} | {'---':>12} | {ols_mse:>10.4f} | {'20':>15} / 20")

---
# Summary

## The Three Regularization Methods

| Method | Penalty | Formula | Key Property |
|---|---|---|---|
| **Ridge** | L2 | $\alpha \sum w_j^2$ | Shrinks coefficients, never zeros them |
| **Lasso** | L1 | $\alpha \sum |w_j|$ | Can zero out coefficients (feature selection) |
| **Elastic Net** | L1 + L2 | $\alpha [\rho \sum |w_j| + \frac{1-\rho}{2} \sum w_j^2]$ | Combines both behaviors |

## Hyperparameters

| Parameter | Role |
|---|---|
| $\alpha$ | Overall regularization strength (higher = more regularization) |
| `l1_ratio` ($\rho$) | Elastic Net only: mix between L1 and L2 (1 = Lasso, 0 = Ridge) |

## When to Use Each Method

| Scenario | Recommended Method |
|---|---|
| Many features, all potentially relevant | **Ridge** — shrinks without discarding |
| Many features, only a few are relevant | **Lasso** — automatic feature selection |
| Correlated features + need for sparsity | **Elastic Net** — groups correlated features |
| Baseline comparison | **OLS** — no regularization |

**Remember:** Always **standardize your features** before applying regularization, since the penalty treats all coefficients equally regardless of feature scale.